In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:

import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt

from Python_arq import engines as engs
from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path




In [3]:
eng = engs.get_engine()
text_qry = text(engs.load_query('qry_olos.sql'))
df = pd.read_sql(text_qry, eng)

In [4]:
dia_semana = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['day_week'] = df['data'].apply(lambda x: day_name[x.weekday()]).map(dia_semana)
df['data'] = pd.to_datetime(df['data'])
df['semana_mes'] = (
    df['data'].dt.day.sub(1) // 7 + 1 
)
df['day_week'] = df['day_week'] + '_W' + df['semana_mes'].astype(str)

In [13]:
## performance bases | Hoje ## 

df_today = df.groupby(['area','base_type']).agg(
    Total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_today['tx_loc'] = (df_today['total_atendidas'] / df_today['Total_tentativas'] * 100).round(2)
df_today = df_today.sort_values('tx_loc', ascending=False)
df_today = df_today[df_today['tx_loc'] >= 2.30]
df_today

,area,base_type,Total_tentativas,total_atendidas,tx_loc
3,Enfermagem,evento,3681,127,3.45
14,Medicina,evento,912,27,2.96
20,Nutrição,evento,11956,347,2.90
27,Psicologia,Base Leads,11742,329,2.80
18,Multi,evento,1487,41,2.76
33,Psiquiatria,evento,958,26,2.71
1,Enfermagem,Material Rico,4377,113,2.58
17,Multi,ativa,488,12,2.46
8,Fisioterapia,evento,10412,252,2.42
0,Enfermagem,Base Leads,2389,57,2.39


In [6]:
## top10 bases ## 
df_top10 = df.groupby(['area','base_type']).agg(
    total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_top10 = df_top10[df_top10['total_tentativas']>=1000]
df_top10['performance'] = (df_top10['total_atendidas'] / df_top10['total_tentativas'] * 100).round(2)
df_top10 = df_top10.sort_values('performance', ascending=False).head(10)
df_top10

,area,base_type,total_tentativas,total_atendidas,performance
28,Multi,evento,11016,470,4.27
41,Pediatria,evento,3797,160,4.21
26,Multi,Material Rico,3556,149,4.19
37,Pediatria,Base Leads,1222,50,4.09
5,Enfermagem,evento,30931,1129,3.65
0,Enfermagem,Base Leads,20262,729,3.60
7,Enfermagem,legado,13305,448,3.37
8,Fisioterapia,Base Leads,32847,1077,3.28
48,Psicologia,evento,17812,584,3.28
33,Nutrição,evento,24514,790,3.22


In [7]:
df

,area,base_type,campaign_id,tablename,data,hour,tentativas,atendidas,day_week,semana_mes
0,Fisioterapia,cancelados,1605,_1605_FISIOTERAPIA_CANCELADO_0_27022026_V513_pt5,2026-02-27,18.0,31,1,Sexta_W4,4
1,Psicologia,ativa,1299,_1299_PSICOLOGIA_ATIVA_OFERTARPRONEUROPSI_0704...,2026-04-07,11.0,222,11,Terça_W1,1
2,Enfermagem,cancelados,1605,_1605_ENFERMAGEM_CANCELADO_16032026_V500_PT1,2026-03-16,10.0,739,5,Segunda_W3,3
3,Enfermagem,inativa,1605,_1605_ENFERMAGEM_INATIVO_0_08012026_V10563,2026-01-09,11.0,27,1,Sexta_W2,2
4,Outras Áreas,None,1553,None,2026-01-28,10.0,10,3,Quarta_W4,4
...,...,...,...,...,...,...,...,...,...,...
9583,Fisioterapia,Base Leads,1605,_1605_FISIOTERAPIA_POS_BASELEADS_09122025_V3314,2025-12-10,11.0,214,6,Quarta_W2,2
9584,Fisioterapia,cancelados,1605,_1605_FISIOTERAPIA_CANCELADO_0_27022026_V513_pt3,2026-03-02,11.0,27,1,Segunda_W1,1
9585,Psicologia,Base Leads,1605,_1605_PSICOLOGIA_POSARTMED_BASELEADS_MATRICULA...,2026-04-02,11.0,111,0,Quinta_W1,1
9586,Medicina,inativa,1605,_1605_MEDICINA_INATIVO_0_03022026_V6187,2026-02-05,9.0,353,6,Quinta_W1,1


In [8]:
## performance bases | D-1 ## 
df_ontem = df[df['data'] == '2026-04-27']
df_ontem = df_ontem.groupby(['area','base_type']).agg(
    Total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_ontem['performance'] = (df_ontem['total_atendidas'] / df_ontem['Total_tentativas'] * 100).round(2)
df_ontem = df_ontem.sort_values('performance', ascending=False)
df_ontem

,area,base_type,Total_tentativas,total_atendidas,performance
0,Enfermagem,evento,501,19,3.79
1,Fisioterapia,evento,645,19,2.95
6,Psicologia,Base Leads,1045,28,2.68
3,Multi,evento,119,3,2.52
2,Fisioterapia,inativa,1009,19,1.88
7,Psicologia,inativa,752,11,1.46
5,Outras Áreas,evento,240,3,1.25
4,Outras Áreas,Carrinho,1687,16,0.95


In [9]:
bases_atuadas = df.groupby(['area','base_type','tablename','data']).agg(
    tentativas = ('tentativas','sum'),
    atendidas = ('atendidas','sum')
).reset_index()
bases_atuadas['split_tablename'] = bases_atuadas['tablename'].str.split('_')
bases_atuadas = bases_atuadas.to_excel(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\bases_atuadas.xlsx')

In [10]:
df = df.to_excel(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\tentativas.xlsx')